In [2]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration


In [3]:
train_data=pd.read_csv("samsum-train.csv")
val_data=pd.read_csv("samsum-validation.csv")

In [4]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [5]:
train_data.shape

(14732, 3)

In [7]:
val_data.shape

(818, 3)

In [8]:
train_data=train_data.sample(n=4000,random_state=42).reset_index(drop=True)
val_data=val_data.sample(n=500,random_state=42).reset_index(drop=True)

In [9]:
train_data.shape

(4000, 3)

In [10]:
import re

def clean_data(text):
    text=re.sub(r"\r\n","",text)
    text=re.sub(r"\s+","",text)
    text=re.sub(r"<.*?>","",text)
    text=text.strip().lower()
    return text

In [11]:
train_data["dialogue"]=train_data["dialogue"].apply(clean_data)
train_data["summary"]=train_data["summary"].apply(clean_data)

val_data["dialogue"]=val_data["dialogue"].apply(clean_data)
val_data["summary"]=val_data["summary"].apply(clean_data)


In [12]:
#tokenize
tokenizer=T5Tokenizer.from_pretrained("t5-small")

In [13]:
#raw data=> tokenizer inputs for fine-turning


def tokenize(data):
    inputs=tokenizer(data["dialogue"],padding="max_length",max_length=512,truncation=True)
    targets=tokenizer(data["summary"],padding="max_length",max_length=150,truncation=True)

    inputs["labels"]=targets["input_ids"]
    return inputs

In [14]:
train_dataset=train_data.apply(tokenize,axis=1).tolist()
val_dataset=val_data.apply(tokenize,axis=1).tolist()

In [15]:
train_dataset[0]

{'input_ids': [25208, 10, 107, 23, 55, 2617, 526, 9, 11465, 8048, 2064, 17, 77, 31, 7, 8372, 232, 23, 11841, 17, 6279, 4188, 51, 2632, 8954, 155, 19405, 53, 11275, 15, 17, 10, 7997, 15, 10, 107, 23, 55, 10, 61, 20349, 7, 6, 2780, 23, 31, 162, 138, 23015, 5236, 155, 5, 10, 61, 7997, 15, 10, 2780, 20349, 7, 1161, 10337, 53, 7932, 526, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [16]:
#working our model

model=T5ForConditionalGeneration.from_pretrained("t5-small")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [17]:
import torch

if torch.backends.mps.is_available():
    device=torch.device("mps")
elif torch.cuda.is_available():
    device=torch.device("cuda")
else:
    device=torch.device("cpu")

print("device: ",device)
model.to(device)

device:  cpu


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [18]:
#Training Arguments

training_args=TrainingArguments(
    output_dir="./results",

    num_train_epochs=6,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500
)

In [19]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [ ]:
#train the model
trainer.train()

C:\Users\nisha\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
